# 09 - Subtype Model Selection

This notebook compares the current subtype prototype models and selects the best practical candidates for the research report and future app integration.

The source review package is documented as non-holdout. The current evaluation is repeated stratified validation because the Supabase export does not include original `subject_id`.

## Objective

The goal is not to force deployment of a weak subtype model. The goal is to decide what subtype prediction can honestly support now, and which model family should be carried forward.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_08 = ROOT / "results" / "subtype_repeated_stratified"
RESULTS_09 = ROOT / "results" / "subtype_embedding_model_selection"

## Baseline To Beat

Notebook 08 established the current repeated stratified prototype baseline. We use macro F1 and balanced accuracy because the subtype classes are imbalanced.

In [ ]:
baseline = pd.read_csv(RESULTS_08 / "repeated_stratified_summary.csv")

baseline[["experiment", "balanced_accuracy_mean", "macro_f1_mean"]]

## Candidate Models

The model-selection pass evaluates several classifiers on cached frozen ResNet-18 embeddings. This is much faster than full CNN fine-tuning on CPU and gives a fairer comparison of classifier choices.

In [ ]:
# Run from the repository root if metrics need to be regenerated.
# !python scripts/select_subtype_embedding_models.py

In [ ]:
selection = pd.read_csv(RESULTS_09 / "embedding_model_selection_summary.csv")

selection.sort_values(["task", "macro_f1_mean"], ascending=[True, False])

## Best Model Per Task

The selected model should improve macro F1 without hiding minority-class failure behind high overall accuracy.

In [ ]:
best = (
    selection.sort_values(["task", "macro_f1_mean"], ascending=[True, False])
    .groupby("task", as_index=False)
    .head(1)
)

best[["task", "model", "balanced_accuracy_mean", "macro_f1_mean", "macro_f1_std"]]

## Per-Class Behavior

The per-class summary shows whether the candidate actually recognizes minority subtypes or mostly predicts the majority subtype.

In [ ]:
per_class = pd.read_csv(RESULTS_09 / "embedding_model_selection_per_class.csv")
best_keys = set(zip(best["task"], best["model"]))

per_class[
    per_class.apply(lambda row: (row["task"], row["model"]) in best_keys, axis=1)
]

## Comparison Against Notebook 08

The embedding model-selection step improves the arch task clearly and gives a small improvement for the whorl task.

In [ ]:
previous = {"arch": 0.536848, "whorl": 0.450181}
comparison = best[["task", "model", "balanced_accuracy_mean", "macro_f1_mean"]].copy()
comparison["previous_macro_f1"] = comparison["task"].map(previous)
comparison["macro_f1_change"] = comparison["macro_f1_mean"] - comparison["previous_macro_f1"]

comparison

## Decision

Carry forward these subtype prototype models:

- Arch subtype: frozen ResNet-18 embeddings + balanced logistic regression (`C=1`).
- Whorl subtype: frozen ResNet-18 embeddings + balanced logistic regression (`C=0.1`).

Do not deploy subtype prediction as a high-confidence final classifier yet. Present it as an expert-reviewed prototype, and visibly flag rare whorl subtype predictions for review.

## App Integration Rule

If subtype prediction is added to the Streamlit app, it should run only after the broad classifier predicts `arch` or `whorl`. Loop images should not be sent to the arch/whorl subtype models.